# Toronto Warming Centre Capacity and Staffing Optimization

This notebook simulates arrivals at seven centres and tests the staffing plan from the original team optimization. The plan is supplied as an input.

The reported planning scenarios and bed-expansion comparison are documented in the project README.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
HOURS_OPEN = 12
EXTREME_PROB = 0.12
STAFF_RATIO = 20

CENTRES = pd.DataFrame({
    'centre': ['Elizabeth','George','Scarborough','North York','Spadina','Cecil','Jimmie Simpson'],
    'capacity': [75,30,68,46,22,30,30],
    'arrival_rate': [6.08,2.50,5.52,3.63,1.76,2.21,2.20],
    'extreme_only': [False,False,False,False,False,True,True]
})
CENTRES

## Demand simulation

In [ ]:
def simulate_demand(n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for scenario, mult in [('Moderate',1.0),('Extreme',1.10)]:
        for _, r in CENTRES.iterrows():
            lam = r.arrival_rate * HOURS_OPEN * mult
            if scenario == 'Moderate' and r.extreme_only:
                lam = 0
            for x in rng.poisson(lam, n):
                rows.append({'scenario':scenario,'centre':r.centre,'demand':int(x)})
    return pd.DataFrame(rows)

demand_sim = simulate_demand()
demand_sim.groupby(['scenario','centre'])['demand'].agg(['mean','std']).round(2)

## Staffing allocation

In [ ]:
MILP_STAFF = {
    'Elizabeth':4,'George':2,'Scarborough':4,'North York':3,
    'Spadina':2,'Cecil':2,'Jimmie Simpson':2
}

staff_check = CENTRES[['centre','capacity']].copy()
staff_check['staff'] = staff_check['centre'].map(MILP_STAFF)
staff_check['staff_supported_capacity'] = staff_check['staff'] * STAFF_RATIO
staff_check['effective_capacity'] = staff_check[['capacity','staff_supported_capacity']].min(axis=1)
staff_check

## 5,000-night stress test

In [ ]:
def evaluate_policy(capacity, staff, n=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    records=[]
    for night in range(n):
        extreme = rng.random() < EXTREME_PROB
        overflow=vacancy=served_local=total_demand=0
        for _, r in CENTRES.iterrows():
            if (not extreme) and r.extreme_only:
                demand=0
            else:
                mult=1.10 if extreme else 1.0
                demand=rng.poisson(r.arrival_rate*HOURS_OPEN*mult)
            eff=min(capacity[r.centre],staff[r.centre]*STAFF_RATIO)
            total_demand += demand
            served_local += min(demand,eff)
            overflow += max(demand-eff,0)
            vacancy += max(eff-demand,0)
        transferred=min(overflow,vacancy)
        turnaways=overflow-transferred
        served=served_local+transferred
        records.append({'night':night+1,'scenario':'Extreme' if extreme else 'Moderate',
                        'demand':total_demand,'served':served,'transferred':transferred,
                        'turnaways':turnaways,'service_level':served/total_demand if total_demand else 1.0})
    return pd.DataFrame(records)

base_caps = dict(zip(CENTRES.centre,CENTRES.capacity))
stress = evaluate_policy(base_caps,MILP_STAFF)
stress[['demand','turnaways','service_level']].describe(percentiles=[.5,.9,.95]).round(3)

## Interpretation

Each site has enough staff for its current beds under the assumed 1:20 ratio. The simulation pools vacancies across sites. On moderate nights, it sets local arrivals at the two extreme-only sites to zero but still counts their beds for transfers. If those sites are closed, moderate-night turn-aways may be understated.

## Transfer eligibility on moderate nights

This paired sensitivity test holds every simulated arrival and the staffing plan fixed. It excludes beds at extreme-only sites from the moderate-night transfer pool. With unrestricted transfers among eligible sites, unmet demand is the excess of total demand over total effective capacity.

In [ ]:
effective = staff_check.set_index('centre')['effective_capacity']
open_moderate = CENTRES.loc[~CENTRES['extreme_only'], 'centre']
moderate_beds = effective.loc[open_moderate].sum()
assert (stress['turnaways'] == (stress['demand'] - effective.sum()).clip(lower=0)).all()

capacity = np.where(stress['scenario'].eq('Extreme'), effective.sum(), moderate_beds)
comparison = stress[['scenario', 'demand', 'turnaways']].copy()
comparison['turnaways_open_sites_only'] = (comparison['demand'] - capacity).clip(lower=0)
comparison['any_unserved_open_sites'] = comparison['turnaways_open_sites_only'].gt(0)
summary = comparison.groupby('scenario').agg(
    nights=('demand', 'size'),
    original_mean=('turnaways', 'mean'),
    open_sites_mean=('turnaways_open_sites_only', 'mean'),
    open_sites_p90=('turnaways_open_sites_only', lambda values: values.quantile(.90)),
    open_sites_nights_with_unmet_demand=('any_unserved_open_sites', 'sum'),
)
summary.round(2)

## Budget reconciliation

The original team report (pp. 6–9) budgets fixed shifts and probability-weighted operating costs. The extreme-night cash requirement is also relevant to an operator paying invoices each night.

In [ ]:
from pathlib import Path

results = Path('results') if Path('results').is_dir() else Path('../results')
assumptions = pd.read_csv(results / 'model_assumptions.csv').set_index('parameter').value
shifts = pd.read_csv(results / 'staffing_results.csv').query("centre != 'Total'").staff.sum()
ops = pd.read_csv(results / 'scenario_results.csv').set_index('scenario').operations_cost
payroll = shifts * assumptions['Staff Wage per 12-hour shift']
expected = payroll + sum(ops[s] * assumptions[f'Probability {s}'] for s in ('Moderate', 'Extreme'))
pd.Series({'Fixed payroll': payroll, 'Expected nightly spend': expected,
           'Expected budget remaining': assumptions['Total Budget'] - expected,
           'Extreme-night spend': payroll + ops['Extreme']}).round(2)